In [1]:
from IPython.display import Markdown, display

with open("ARCHITECTURE.md", "r", encoding="utf-8") as f:
    content = f.read()

display(Markdown(content))

# Architecture — LILA BLACK Level Designer Tool

## Tech Stack

| Layer | Choice | Why |
|-------|--------|-----|
| Frontend + Backend | Streamlit (Python) | Zero frontend experience needed; pure Python; deploys in 2 min on Streamlit Cloud |
| Data loading | PyArrow + Pandas | PyArrow reads extensionless parquet natively; Pandas for filtering/groupby |
| Visualization | Plotly (graph_objects) | Interactive zoom/pan on minimap; supports layout_image background + Heatmap overlay in same figure |
| Image handling | Pillow | Opens .png and .jpg minimaps, passes directly to Plotly layout_image |
| Hosting | Streamlit Cloud | Free, GitHub-connected, no infra config |

## Data Flow

```
player_data/February_*/        (1,243 extensionless parquet files)
         │
         ▼
data_loader.py::load_all_data()
  ├── PyArrow reads each file
  ├── Decode event bytes → string
  ├── Detect human vs bot (UUID vs numeric user_id)
  ├── Strip ".nakama-0" from match_id
  ├── Compute ts_relative (seconds from match start, per match_id group)
  └── Pre-compute pixel_x, plot_y for every row (vectorized, per map)
         │
         ▼  st.cache_data (cached in memory for session)
         │
         ▼
app.py — Sidebar filters (map → date → match → player type)
  │
  ├── Tab 1: Player Journeys
  │     ├── go.Scatter (paths): None-separator trick → 1 trace per player type
  │     └── go.Scatter (events): 1 trace per event type, pre-computed coords
  │
  ├── Tab 2: Heatmaps
  │     ├── np.histogram2d on pixel_x, plot_y
  │     └── go.Heatmap overlay on minimap background
  │
  └── Tab 3: Stats
        ├── Metrics, bar charts, per-match table
        └── Event timeline (15s bins, stacked bar)
```

## Coordinate Mapping

The tricky part — two coordinate flips needed:

**Step 1: World → image pixel**
```
u        = (x - origin_x) / scale
v        = (z - origin_z) / scale
pixel_x  = u * 1024
pixel_y  = (1 - v) * 1024   ← flipped: image origin is top-left (y=0 = top)
```

**Step 2: Image pixel → Plotly coordinate**
```
plot_y = 1024 - pixel_y     ← flipped again: Plotly y=0 is at bottom
```

The minimap image is placed in Plotly at `x=0, y=1024` (top-left corner in Plotly coords).  
After double-flip, scatter points align correctly with the background image.

## Assumptions Made

| Ambiguity | Assumption |
|-----------|------------|
| ts column semantics | "Milliseconds elapsed in match" appears to be wall-clock timestamps; treated as sortable per-match timestamps, only relative ordering matters |
| Minimap image size | Assumed 1024×1024 as stated in README, not verified per image |
| Multi-map files | A match is always on one map (match_id never spans maps) |
| Feb 14 partial day | Included as-is; filtered by date in sidebar |

## Trade-offs

| Decision | Trade-off |
|----------|-----------|
| Paths only on single match | Multi-match paths = unreadable noise; heatmaps cover the aggregate case |
| Pre-compute pixel coords at load | ~10% more RAM, but all rendering is instant |
| Streamlit vs React | Streamlit is slower to render large figures but requires zero frontend code |
| No real-time playback animation | Streamlit can't animate smoothly; timeline slider approximates this |
| All data in memory | Fine for 89k rows; would need chunked loading for 10M+ rows |

## With More Time

- Frame-by-frame animated playback (Plotly `animation_frame` or a JS frontend)
- Player-specific highlighting (click a player → highlight their path)
- Extraction point overlays (if data available)
- Storm zone visualization (overlay shrinking circle over time)
- Persistent URL state (share a specific match/filter combination)
- Pre-aggregate heatmaps by map at load time for instant switching
